# LAMA Inpainting Test - Manual Mask Drawing

This notebook demonstrates using the LAMA model with **manually drawn masks**.
Instead of using pre-generated masks, you can interactively draw masks on images using your mouse.

## Requirements
- Images in `../data/sample_images/`
- LAMA model weights in `../models/big-lama/`
- OpenCV for interactive mask drawing

## Process
1. Load LAMA model
2. For each image, open an interactive window
3. Draw mask with mouse (white brush)
4. Press 's' to save and proceed, 'r' to reset, 'q' to quit
5. Run inpainting inference
6. Display and save results

In [ ]:
import os
import sys
import torch
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from omegaconf import OmegaConf
import numpy as np
import cv2

# Add parent directory to path for saicinpainting imports
sys.path.insert(0, os.path.abspath('..'))
from saicinpainting.training.trainers import load_checkpoint
from saicinpainting.evaluation.data import pad_img_to_modulo

# PATHS 
model_dir = "../models/big-lama"
config_path = os.path.join(model_dir, "config.yaml")
checkpoint_path = os.path.join(model_dir, "models/best.ckpt")

img_dir = "../data/sample_images"
out_dir = "../data/output_manual"
os.makedirs(out_dir, exist_ok=True)

# Temporary experiment & tensorboard dirs
exp_dir = "../temp/experiments"
tb_dir = "../temp/tb_logs"
os.makedirs(exp_dir, exist_ok=True)
os.makedirs(tb_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
train_config = OmegaConf.load(config_path)

train_config.location.out_root_dir = exp_dir
train_config.location.tb_dir = tb_dir
train_config.visualizer.outdir = os.path.join(exp_dir, "samples")
train_config.data.data_root_dir = "../data"


train_config.losses.resnet_pl.weights_path = os.path.expanduser("~/.cache/torch/ade20k/ade20k-resnet50dilated-ppm_deepsup")

# LOAD MODEL 
model = load_checkpoint(train_config, checkpoint_path, map_location=device, strict=False)
model.eval()
model.to(device)
print("Loaded model checkpoint")

In [ ]:
def draw_mask_on_image(image_path):
    """Open an image and allow the user to draw a white mask using the mouse."""
    img = cv2.imread(image_path)
    mask = np.zeros(img.shape[:2], np.uint8)

    drawing = False
    brush_size = 15
    color = (255, 255, 255)

    def draw(event, x, y, flags, param):
        nonlocal drawing
        if event == cv2.EVENT_LBUTTONDOWN:
            drawing = True
        elif event == cv2.EVENT_MOUSEMOVE:
            if drawing:
                cv2.circle(img, (x, y), brush_size, color, -1)
                cv2.circle(mask, (x, y), brush_size, 255, -1)
        elif event == cv2.EVENT_LBUTTONUP:
            drawing = False

    cv2.namedWindow("Draw Mask (press 's' to save, 'r' to reset, 'q' to quit)")
    cv2.setMouseCallback("Draw Mask (press 's' to save, 'r' to reset, 'q' to quit)", draw)

    while True:
        display = cv2.addWeighted(img, 0.7, cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR), 0.3, 0)
        cv2.imshow("Draw Mask (press 's' to save, 'r' to reset, 'q' to quit)", display)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('s'):  # Save
            cv2.destroyAllWindows()
            return mask
        elif key == ord('r'):  # Reset
            mask[:] = 0
            img = cv2.imread(image_path)
        elif key == ord('q'):  # Quit
            cv2.destroyAllWindows()
            return None

In [ ]:
# IMAGE TRANSFORMS 
to_tensor = transforms.ToTensor()
to_pil = transforms.ToPILImage()

#RUN INFERENCE 
for fname in tqdm(os.listdir(img_dir), desc="Inpainting images"):
    if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    img_path = os.path.join(img_dir, fname)
    mask_array = draw_mask_on_image(img_path)
    if mask_array is None:
        print("Skipping this image.")
        continue

    # Load image and mask
    image = to_tensor(Image.open(img_path).convert("RGB"))
    mask = torch.from_numpy(mask_array / 255.0).unsqueeze(0).float()

    # Pad both to multiple of 8
    image = pad_img_to_modulo(image, 8)
    mask = pad_img_to_modulo(mask, 8)

    # Convert back to tensors if numpy arrays
    if isinstance(image, np.ndarray):
        image = torch.from_numpy(image)
    if isinstance(mask, np.ndarray):
        mask = torch.from_numpy(mask)

    # Add batch dimension
    batch = {
        "image": image.unsqueeze(0).to(device),
        "mask": mask.unsqueeze(0).to(device)
    }

    # Run inference
    with torch.no_grad():
        result = model(batch)
        inpainted = result['inpainted'][0].cpu()

    # Display results
    print(f"\nProcessed: {fname}")
    print("Displaying: Original -> Mask -> Inpainted")
    to_pil(image).show()
    to_pil(mask).show()
    to_pil(inpainted).show()

    # Save result
    out_path = os.path.join(out_dir, fname)
    to_pil(inpainted).save(out_path)
    print(f"Saved to: {out_path}")

print(f"\nInpainting complete! Results saved in: {out_dir}")